# Retrieval: dense, lexical, and RRF-fused — does fusion help?

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/f-inverse/jammi-ai/blob/py-v0.49.1/cookbook/notebooks/book/10-retrieval/retrieval.ipynb)

Built from [`cookbook/book/chapters/10-retrieval/retrieval.qmd`](https://github.com/f-inverse/jammi-ai/blob/main/cookbook/book/chapters/10-retrieval/retrieval.qmd). Run the setup
cell first; every other cell runs top to bottom.

In [ ]:
# Setup: jammi 0.49.1 — the CUDA engine on an sm_80+ GPU (L4, A100, …), the
# CPU engine otherwise — and the cookbook's library and fixtures. On that GPU the
# chapter runs at `full` scale, over the published data; set SCALE = "small" to
# run the seconds-long version over the committed fixtures instead.
import os
import subprocess
import sys


def compute_capability() -> float:
    try:
        out = subprocess.run(
            ["nvidia-smi", "--query-gpu=compute_cap", "--format=csv,noheader"],
            capture_output=True, text=True, check=True,
        ).stdout.split()
    except (OSError, subprocess.CalledProcessError):
        return 0.0
    return float(out[0]) if out else 0.0


gpu = compute_capability() >= 8.0
engine = "jammi-ai-native-cu12" if gpu else "jammi-ai-native"
server = "jammi-server-cu12" if gpu else "jammi-server"
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "jammi-ai==0.49.1", engine + "==0.49.1", "jammi-cookbook==0.49.1"], check=True)
SCALE = "full" if gpu else "small"
os.environ["JAMMI_COOKBOOK_SCALE"] = SCALE
print(f"engine: {engine}   scale: {SCALE}")

In [ ]:
import jammi_cookbook

**Recipe:** `encode_query` + `search(embedding_table=…)` ·
`build_lexical_index` + `lexical_search` · `rrf_fuse` ·
`assemble_context(hybrid=True)` · **Theory:** dense vs sparse retrieval,
reciprocal rank fusion (Cormack et al. 2009), the BM25 probabilistic-relevance
baseline (Robertson & Zaragoza 2009) · **Rail:** measurement (per-method
precision\@10 / nDCG\@10 and the honest fusion delta).

The keystone measured one retrieval number per embedding table. This chapter asks
the question a practitioner asks: **given dense embeddings, a lexical scorer, and
a graph-propagated variant, which retriever — and which fusion — should I reach
for?** The task is fixed (same-subject retrieval over the papers, asked by title,
scored against tier 01's embedding-independent golden) and the *retriever* varies.

In [ ]:
import math
import tempfile
from collections import defaultdict

import jammi
from jammi_cookbook import contracts, datasets, encoders, keystone, scale

SCALE = scale.current()
MODEL = encoders.text(SCALE)
DEPTH = 100  # candidates per ranked list before fusion

db = jammi.connect(f"file://{tempfile.mkdtemp()}")
arxiv = datasets.arxiv(db, SCALE)
raw = keystone.embed(db, arxiv, SCALE)
propagated = keystone.propagate(db, arxiv, raw)
golden = keystone.subject_golden(db, arxiv)

relevant: dict[str, set[str]] = defaultdict(set)
query_text: dict[str, str] = {}
for g in db.sql(f"SELECT query_id, query_text, relevant_id FROM {golden}").to_pylist():
    relevant[g["query_id"]].add(g["relevant_id"])
    query_text[g["query_id"]] = g["query_text"]
queries = sorted(relevant)
print(f"{len(queries)} queries, each asked by its title")

## Dense retrieval: one `search` per table

A source can carry several embedding tables — here the raw embeddings and the
graph-propagated ones. `search` takes the query vector and, with
`embedding_table=`, the table to search; without it, the source's current
default. The query is encoded once, by the model that produced both tables'
space (propagation keeps it).

In [ ]:
query_vectors = {q: db.encode_query(model=MODEL, query=query_text[q]) for q in queries}


def dense(table: str):
    def rank(q: str) -> list[str]:
        hits = db.search(arxiv.papers, query=query_vectors[q], k=DEPTH, embedding_table=table)
        return hits.column("_row_id").to_pylist()

    return rank


def score(ranked: dict[str, list[str]]) -> tuple[float, float]:
    """precision@10 and nDCG@10 against the same-subject golden."""
    precision = ndcg = 0.0
    for q, ranks in ranked.items():
        hits = [1.0 if r in relevant[q] else 0.0 for r in ranks[:10]]
        ideal = sum(1 / math.log2(i + 2) for i in range(min(10, len(relevant[q]))))
        precision += sum(hits) / 10
        ndcg += sum(h / math.log2(i + 2) for i, h in enumerate(hits)) / ideal
    return precision / len(ranked), ndcg / len(ranked)


rankers = {"dense_raw": dense(raw), "dense_prop": dense(propagated)}

## Lexical BM25 — the sparse arm

BM25 (Robertson & Zaragoza 2009) is the probabilistic-relevance lexical score — term
frequency saturated and length-normalized, weighted by inverse document
frequency. It scores the *same* papers against the *same* relevance target, from
the query's words alone. `build_lexical_index` materialises the papers' text as a
lexical table — one row per paper, its title and abstract joined — and
`lexical_search` ranks it against the query's words, hydrating the papers the
way `search` does, each hit carrying its `bm25_score` and `bm25_rank`.

In [ ]:
lexical = db.build_lexical_index(arxiv.papers, columns=["title", "abstract"], key="paper_id")


def bm25(q: str) -> list[str]:
    hits = db.lexical_search(arxiv.papers, text=query_text[q], k=DEPTH, lexical_table=lexical)
    return hits.column("_row_id").to_pylist()


example = db.lexical_search(arxiv.papers, text=query_text[queries[0]], k=3, lexical_table=lexical)
print(example.select(["_row_id", "bm25_score", "bm25_rank", "retrieved_by"]).to_pylist())
rankers["bm25"] = bm25

The analyzer stems English (`"raw"` would not), and the query's words are
terms, not syntax: a title's colon or quote is text.

## RRF fusion — the engine's `rrf_fuse`

Reciprocal rank fusion (Cormack et al. 2009) combines ranked lists by summing
`1 / (k_rrf + rank)` across rankers — rank-only, so it needs no score
calibration. We fuse the two dense arms, and the **hybrid** dense + lexical.

In [ ]:
ranked = {name: {q: rank(q) for q in queries} for name, rank in rankers.items()}
ranked["rrf_raw_prop"] = {
    q: [doc for doc, _ in db.rrf_fuse([ranked["dense_raw"][q], ranked["dense_prop"][q]])]
    for q in queries
}
ranked["rrf_raw_bm25"] = {
    q: [doc for doc, _ in db.rrf_fuse([ranked["dense_raw"][q], ranked["bm25"][q]])]
    for q in queries
}

metrics = {name: score(r) for name, r in ranked.items()}
print(f"{'method':<14}{'precision@10':>14}{'nDCG@10':>10}")
for name, (p, n) in metrics.items():
    print(f"{name:<14}{p:>14.3f}{n:>10.3f}")

In [ ]:
for name, (p, n) in metrics.items():
    contracts.assert_close(f"retrieval.{name}.precision_at_10", p, tol=0.02)
    contracts.assert_close(f"retrieval.{name}.ndcg_at_10", n, tol=0.02)

## The honest fusion finding

This is the part a benchmark table would quietly drop.

In [ ]:
best_single = max(metrics[m][0] for m in ("dense_raw", "dense_prop", "bm25"))
best_fused = max(metrics[m][0] for m in ("rrf_raw_prop", "rrf_raw_bm25"))
print(f"best single arm: {best_single:.3f}   best fusion: {best_fused:.3f}   "
      f"fusion helps: {best_fused > best_single}")

In [ ]:
if SCALE is scale.Scale.FULL:
    assert best_fused <= best_single  # no fused list beats the dominant single arm

At `full` scale **fusion does not help**: RRF of the two dense arms beats the
*weaker* raw arm but sits below the propagated arm, and the hybrid dense + lexical
list is dragged down by the weaker lexical ranker. The reason is structural:
**RRF cannot exceed the best single ranker it contains when one ranker
dominates** — mixing in a weaker list only dilutes the strong list's top ranks.
Fusion pays off when the arms are *complementary and comparably strong*; here
the propagated arm already dominates, so the strongest move is to use it alone —
the lesson tier 02 taught from the other side: **graph propagation, not fusion,
is the lever on this target.** Measure each arm first; fuse only when a weaker
arm contributes relevant documents the strong arm misses, and confirm the fused
list actually beats the best single arm.

## `assemble_context(hybrid=True)` — a context from similarity and citations

Retrieval also feeds prediction. `assemble_context` retrieves a target's context
set and pools it into one vector; `hybrid=True` takes the union of its
embedding-similarity neighbours and a declared-edge walk (here, the citations),
and reports how the set was assembled.

In [ ]:
target = queries[0]
context = db.assemble_context(
    arxiv.papers, query=query_vectors[target], k=5, exclude_key=target,
    edge_source=arxiv.cites, edge_src_column="src", edge_dst_column="dst", edge_hops=1,
    hybrid=True,
)
print(f"assembled by: {context['source']}   members: {context['context_size']}")
print("context keys:", context["context_keys"][:8])

In [ ]:
assert context["source"] == "hybrid" and context["context_size"] >= 5

In [ ]:
db.close()

## Bridge note

> **Retrieval is one primitive at three resolutions.** A dense kNN, a lexical
> BM25 scan, and a graph-propagated kNN all *rank documents by a relevance
> signal*; RRF (Cormack et al. 2009) is a rank-only combiner over those lists. The
> measured spread says the *signal* — dense-semantic vs lexical vs
> graph-propagated — is the large lever and the combiner the small one.

## References

- Cormack, Gordon V., Clarke, Charles L. A., Büttcher, Stefan (2009) *Reciprocal Rank Fusion Outperforms Condorcet and Individual Rank Learning Methods* Proceedings of the 32nd International ACM SIGIR Conference on Research and Development in Information Retrieval DOI 10.1145/1571941.1572114.
- Robertson, Stephen, Zaragoza, Hugo (2009) *The Probabilistic Relevance Framework: BM25 and Beyond* Foundations and Trends in Information Retrieval DOI 10.1561/1500000019.